## 1. Setup and Imports


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import KFold, train_test_split, ParameterGrid

from opfython.models import SupervisedOPF
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
)

from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC as SupportVectorMachineClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight

from imblearn.under_sampling import NearMiss

from scipy.stats import wilcoxon
import scipy.stats as st
from collections import Counter

import logging
import warnings


## 2. Data Loading and Initial Exploration


In [ ]:
df = pd.read_csv('NPHA-doctor-visits.csv')


In [ ]:
df.head()


In [ ]:
df.shape


In [ ]:
df.isna().sum()


In [ ]:
df.info()


In [ ]:
df.describe()


## 3. Exploratory Data Analysis


In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(3, 5, figsize=(20, 15))
axes = axes.ravel()

for i, column in enumerate(df.columns):
    df[column].value_counts().plot(kind='bar', ax=axes[i])
    axes[i].set_title(column)

plt.tight_layout()
plt.show()


In [ ]:
# Target variable statistics
df['Number of Doctors Visited'].describe()


In [ ]:
# Age variable — constant across dataset
df['Age'].describe()


In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(y='Number of Doctors Visited', data=df)
plt.title("Boxplot of Number of Doctors Visited")
plt.show()


In [ ]:
# Quartile analysis for the target variable
Q1_target = df['Number of Doctors Visited'].quantile(0.25)
Q2_target = df['Number of Doctors Visited'].quantile(0.5)
Q3_target = df['Number of Doctors Visited'].quantile(0.75)

lower_limit = Q1_target - 1.5 * (Q3_target - Q1_target)
upper_limit = Q3_target + 1.5 * (Q3_target - Q1_target)

print(f"Feature: Number of Doctors Visited")
print(f"- Q1: {Q1_target:.2f}")
print(f"- Q2 (Median): {Q2_target:.2f}")
print(f"- Q3: {Q3_target:.2f}")
print(f"- Lower Limit: {lower_limit:.2f}")
print(f"- Upper Limit: {upper_limit:.2f}")


Class 2 (2 visits) is over-represented. Downsampling the majority class may be beneficial.


In [ ]:
# Class distribution
class_distribution = df['Number of Doctors Visited'].value_counts()
print(class_distribution)


In [ ]:
# Demographics exploration with readable labels
df_labeled = df.copy()

age_dict = {1: "50-64", 2: "65-80"}
df_labeled['Age'] = df_labeled['Age'].map(age_dict)

race_dict = {
    1: "White, Non-Hispanic", 2: "Black, Non-Hispanic",
    3: "Other, Non-Hispanic", 4: "Hispanic", 5: "2+ Races, Non-Hispanic"
}
df_labeled['Race'] = df_labeled['Race'].map(race_dict)

gender_dict = {1: "Male", 2: "Female"}
df_labeled['Gender'] = df_labeled['Gender'].map(gender_dict)

fig, axes = plt.subplots(1, 3, figsize=(20, 10))
for i, col in enumerate(['Age', 'Race', 'Gender']):
    counts = df_labeled[col].value_counts()
    axes[i].bar(counts.index.astype(str), counts.values,
                color=sns.color_palette("viridis", len(counts)))
    axes[i].set_title('Distribution of ' + col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    # Annotate bars
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap
corr_matrix = df.corr()

plt.figure(figsize=(18, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=True, fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix")
plt.show()


## 4. Data Preprocessing


**Target encoding**: Binarize the target — replace value 3 with 2 (merge high-frequency classes).


In [ ]:
# Binarize target: merge class 3 into class 2
df['Number of Doctors Visited'] = df['Number of Doctors Visited'].replace(3, 2)


**One-Hot Encoding**: All features are categorical, so OHE is applied to all columns.


In [ ]:
# One-Hot Encode all categorical features
categorical_features = df.columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=False)
df_encoded.head()


In [ ]:
df_encoded = df_encoded.reset_index(drop=True)


In [ ]:
# Separate features and target
X = df_encoded.drop(columns=['Number of Doctors Visited_1', 'Number of Doctors Visited_2'])
y = df_encoded[['Number of Doctors Visited_1', 'Number of Doctors Visited_2']]


**Feature selection**: Drop low-importance and redundant features identified in prior analysis.


In [ ]:
# Low-importance features to remove
low_importance_features = [
    'Race_5', 'Uknown Keeps Patient from Sleeping_0', 'Prescription Sleep Medication_2',
    'Phyiscal Health_1', 'Mental Health_-1', 'Race_4', 'Dental Health_1',
    'Prescription Sleep Medication_-1', 'Dental Health_-1', 'Mental Health_5',
    'Phyiscal Health_-1', 'Age_2', 'Gender_1', 'Phyiscal Health_3', 'Phyiscal Health_5',
    'Dental Health_3', 'Race_2', 'Trouble Sleeping_3', 'Mental Health_3', 'Gender_2',
    'Employment_1', 'Trouble Sleeping_2', 'Dental Health_6', 'Trouble Sleeping_-1'
]

X = X.drop(columns=low_importance_features)


In [ ]:
print(f"Dimensions after feature selection: {X.shape}")


In [ ]:
# Convert to numpy arrays
X_features = X.values
y_onehot = y.values


In [ ]:
# Convert one-hot target to single-column labels
y = np.argmax(y_onehot, axis=1)
y = pd.Series(y)

class_distribution = y.value_counts()
print("Class distribution:")
print(class_distribution)


In [ ]:
# Class distribution bar plot
ax = y.value_counts().plot(kind="bar", title="Number of Doctors Visited",
                            color=sns.color_palette("viridis", len(y.value_counts())))
ax.set_xlabel("Classes")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.text(p.get_x() + p.get_width()/2, p.get_height() + 0.5,
            str(int(p.get_height())), ha='center', fontweight='bold')
plt.show()


In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## 5. PCA Visualization


In [ ]:
# 2D PCA
pca = PCA(n_components=2)
principal_components = pca.fit_transform(X_scaled)
df_pca = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])
df_pca['Target'] = y


In [ ]:
# 2D PCA scatter plot (simple)
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='PC1', y='PC2', hue='Target',
    palette='viridis', data=df_pca,
    alpha=0.8, s=70
)


In [ ]:
# 2D PCA scatter plot (annotated)
sns.scatterplot(x='PC1', y='PC2', hue='Target',
                palette='viridis', data=df_pca, alpha=0.8, s=70)
expl_var_1 = pca.explained_variance_ratio_[0] * 100
expl_var_2 = pca.explained_variance_ratio_[1] * 100
plt.title('Principal Components in 2D (PCA Plot)', fontsize=15)
plt.xlabel(f'Principal Component 1 ({expl_var_1:.2f}% explained variance)')
plt.ylabel(f'Principal Component 2 ({expl_var_2:.2f}% explained variance)')
plt.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# 3D PCA
pca_3d = PCA(n_components=3)
components_3d = pca_3d.fit_transform(X_scaled)

df_pca3 = pd.DataFrame(data=components_3d, columns=['PC1', 'PC2', 'PC3'])
df_pca3['Target'] = y

fig3d = plt.figure(figsize=(12, 10))
ax = fig3d.add_subplot(111, projection='3d')

colors = ['purple', 'teal', 'gold']
unique_classes = sorted(df_pca3['Target'].unique())

for cls, color in zip(unique_classes, colors):
    subset = df_pca3[df_pca3['Target'] == cls]
    ax.scatter(subset['PC1'], subset['PC2'], subset['PC3'],
               color=color, label=f'Class {cls}', s=60, alpha=0.9,
               edgecolor='black', linewidth=0.5)

e1_3d = pca_3d.explained_variance_ratio_[0] * 100
e2_3d = pca_3d.explained_variance_ratio_[1] * 100
e3_3d = pca_3d.explained_variance_ratio_[2] * 100

ax.set_xlabel(f'PC1 ({e1_3d:.2f}%)', fontweight='bold', labelpad=10)
ax.set_ylabel(f'PC2 ({e2_3d:.2f}%)', fontweight='bold', labelpad=10)
ax.set_zlabel(f'PC3 ({e3_3d:.2f}%)', fontweight='bold', labelpad=10)
plt.title('PCA 3D — Target Dimensional Depth', fontsize=16)
plt.legend(title='Target/Class', loc='best')
ax.view_init(elev=20, azim=135)
plt.tight_layout()
plt.show()


In [ ]:
# Prepare feature matrix for modeling (numpy array of selected features)
X_features_df = pd.DataFrame(X_features, columns=X.columns)
x = X_features_df.values


## 6. Model Configuration


In [ ]:
# Combined hyperparameter grids for all classifiers
param_grids = {
    'KNN': {
        'n_neighbors': [10, 20, 30, 40],
        'metric': ['euclidean', 'manhattan', 'minkowski', 'cosine'],
    },
    'DT': {
        'max_depth': [3, 5, 7],
        'min_samples_split': [10, 20, 30],
        'min_samples_leaf': [5, 10],
    },
    'RF': {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'min_samples_leaf': [2, 5, 10],
    },
    'SVM': {
        'C': [0.1, 1, 10],
        'kernel': ['rbf'],
        'gamma': ['scale', 0.1],
        'probability': [True],
    },
    'MLP': {
        'hidden_layer_sizes': [(32, 16), (64,)],
        'alpha': [0.01, 0.1, 1.0],
        'learning_rate_init': [0.001],
        'activation': ['relu', 'tanh'],
        'solver': ['adam'],
        'max_iter': [2000],
        'early_stopping': [True],
        'random_state': [42],
    },
    'LR': {
        'C': [0.01, 0.1, 1, 10],
        'solver': ['saga'],
        'penalty': ['l1', 'l2'],
    },
    'XGB': {
        'n_estimators': [100, 200],
        'max_depth': [3, 5],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.8],
        'colsample_bytree': [0.8],
        'reg_alpha': [0.1, 1.0],
    },
    'NB': {
        'var_smoothing': [1e-11, 1e-10, 1e-9, 1e-8, 1e-7],
    },
    'NC': {
        'metric': ['euclidean', 'manhattan'],
        'shrink_threshold': [None, 0.1, 0.5, 1.0],
    },
    'OPF': {
        'distance': ['euclidean', 'squared_euclidean', 'log_squared_euclidean',
                     'manhattan', 'canberra', 'chebyshev'],
    },
}


In [ ]:
# 10-fold cross-validation with fixed random state
kfold = KFold(n_splits=10, shuffle=True, random_state=42)


In [ ]:
# MinMaxScaler for model input normalization
minmax_scaler = MinMaxScaler()


## 7. Cross-Validation Initialization


In [ ]:
# Model names and metrics
MODEL_NAMES = ['KNN', 'DT', 'RF', 'SVM', 'MLP', 'LR', 'XGB', 'NB', 'NC', 'OPF']
METRICS = ['accuracy', 'precision', 'recall', 'f1', 'auroc']

# Nested dict: results[model_name][metric] = list of fold scores
results = {model: {metric: [] for metric in METRICS} for model in MODEL_NAMES}
confusion_matrices = {model: [] for model in MODEL_NAMES}


In [ ]:
# MDI (Mean Decrease in Impurity) results — only for tree-based models
mdi_results = {
    'DT': [], 'RF': [], 'XGB': [], 'OPF': []
}


In [ ]:
# GridSearch best-score records (per fold)
gridsearch_results_f1 = {model: [] for model in MODEL_NAMES}
gridsearch_results_acc = {model: [] for model in MODEL_NAMES}


In [ ]:
# Validation metric: macro-averaged F1 score
def validation_metric(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

# AUROC validation metric (for models that support predict_proba)
def validation_metric_auroc(y_true, y_proba):
    return roc_auc_score(y_true, y_proba)


## 8. Cross-Validation Training Loop


In [ ]:
# Suppress OPF logging and warnings for clean output
logging.getLogger("opfython").disabled = True
logging.disable(logging.CRITICAL)
warnings.filterwarnings("ignore")


In [ ]:
# Model specifications: class constructor and base fit keyword arguments
MODEL_CONFIGS = {
    'KNN': {
        'class': KNeighborsClassifier,
        'grid_fit_kwargs': {'weights': 'distance'},
        'final_fit_kwargs': {'weights': 'distance'},
        'supports_proba': True,
        'tracks_mdi': False,
        'is_opf': False,
    },
    'DT': {
        'class': DecisionTreeClassifier,
        'grid_fit_kwargs': {'class_weight': 'balanced', 'random_state': 42},
        'final_fit_kwargs': {'class_weight': 'balanced', 'random_state': 42},
        'supports_proba': True,
        'tracks_mdi': True,
        'is_opf': False,
    },
    'RF': {
        'class': RandomForestClassifier,
        'grid_fit_kwargs': {'class_weight': 'balanced', 'random_state': 42},
        'final_fit_kwargs': {'class_weight': 'balanced', 'random_state': 42},
        'supports_proba': True,
        'tracks_mdi': True,
        'is_opf': False,
    },
    'SVM': {
        'class': SupportVectorMachineClassifier,
        'grid_fit_kwargs': {'probability': True, 'class_weight': 'balanced', 'random_state': 42},
        'final_fit_kwargs': {'probability': True, 'class_weight': 'balanced', 'random_state': 42},
        'supports_proba': True,
        'tracks_mdi': False,
        'is_opf': False,
    },
    'MLP': {
        'class': MLPClassifier,
        'grid_fit_kwargs': {'sample_weight': None},   # set dynamically
        'final_fit_kwargs': {'sample_weight': None},  # set dynamically
        'supports_proba': True,
        'tracks_mdi': False,
        'is_opf': False,
    },
    'LR': {
        'class': LogisticRegression,
        'grid_fit_kwargs': {'class_weight': 'balanced', 'max_iter': 1000, 'random_state': 42},
        'final_fit_kwargs': {'class_weight': 'balanced'},
        'supports_proba': True,
        'tracks_mdi': False,
        'is_opf': False,
    },
    'XGB': {
        'class': XGBClassifier,
        'grid_fit_kwargs': {'scale_pos_weight': None},   # set dynamically
        'final_fit_kwargs': {'scale_pos_weight': None},  # set dynamically
        'supports_proba': True,
        'tracks_mdi': True,
        'is_opf': False,
    },
    'NB': {
        'class': GaussianNB,
        'grid_fit_kwargs': {'priors': [0.5, 0.5]},
        'final_fit_kwargs': {'priors': [0.5, 0.5]},
        'supports_proba': True,
        'tracks_mdi': False,
        'is_opf': False,
    },
    'NC': {
        'class': NearestCentroid,
        'grid_fit_kwargs': {},
        'final_fit_kwargs': {},
        'supports_proba': False,  # NC does not support predict_proba
        'tracks_mdi': False,
        'is_opf': False,
    },
    'OPF': {
        'class': SupervisedOPF,
        'grid_fit_kwargs': {},
        'final_fit_kwargs': {},
        'supports_proba': False,  # OPF does not support predict_proba
        'tracks_mdi': True,
        'is_opf': True,
    },
}


In [ ]:
def grid_search_model(model_class, param_grid, X_train, y_train, X_val, y_val,
                      extra_fit_kwargs=None, is_opf=False):
    """
    Run manual grid search over parameter grid.
    Routes 'sample_weight' to .fit() (for MLP). Routes all other keys
    (including 'scale_pos_weight' for XGB) to the model constructor.
    Returns best_params (dict), best_f1 (float), best_acc (float).
    """
    if extra_fit_kwargs is None:
        extra_fit_kwargs = {}

    # Separate fit-only kwargs from constructor kwargs
    fit_kwargs = {}
    constructor_kwargs = {}
    for k, v in extra_fit_kwargs.items():
        if k == 'sample_weight':
            fit_kwargs[k] = v
        else:
            constructor_kwargs[k] = v

    best_params = None
    best_f1 = -1.0
    best_acc = 0.0

    for params in ParameterGrid(param_grid):
        model = model_class(**{**params, **constructor_kwargs})

        if is_opf:
            model.fit(np.array(X_train), np.array(y_train, dtype=np.int32))
            y_pred = model.predict(np.array(X_val))
        else:
            model.fit(X_train, y_train, **fit_kwargs)
            y_pred = model.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        f1 = validation_metric(y_val, y_pred)

        if f1 > best_f1:
            best_f1 = f1
            best_params = params
            best_acc = acc

    return best_params, best_f1, best_acc


In [ ]:
# ============================================================
# Main 10-fold CV training loop
# ============================================================

for i, (train_index, test_index) in enumerate(kfold.split(x)):
    X_train, X_test = x[train_index], x[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Internal validation split (80/20 of training data)
    X_train_sub, X_val, y_train_sub, y_val = train_test_split(
        X_train, y_train, test_size=0.2)

    # --- Cost-Sensitive Learning: per-fold class weights ---
    counts_sub = Counter(y_train_sub)
    counts_full = Counter(y_train)
    sw_train_sub = compute_sample_weight('balanced', y_train_sub)
    sw_train_full = compute_sample_weight('balanced', y_train)

    # Scale data
    X_train_scaled = minmax_scaler.fit_transform(X_train)
    X_test_scaled = minmax_scaler.transform(X_test)
    X_train_sub_scaled = minmax_scaler.fit_transform(X_train_sub)
    X_val_scaled = minmax_scaler.transform(X_val)

    print(f'Fold {i+1}:')

    for model_name in MODEL_NAMES:
        cfg = MODEL_CONFIGS[model_name]

        # --- Grid Search on validation set ---
        grid_fit_kw = cfg['grid_fit_kwargs'].copy()

        # Dynamically set sample_weight for MLP
        if model_name == 'MLP':
            grid_fit_kw['sample_weight'] = sw_train_sub
        # Dynamically set scale_pos_weight for XGB
        if model_name == 'XGB':
            grid_fit_kw['scale_pos_weight'] = counts_sub[0] / counts_sub[1]

        best_params, best_f1, best_acc = grid_search_model(
            cfg['class'], param_grids[model_name],
            X_train_sub_scaled, y_train_sub, X_val_scaled, y_val,
            extra_fit_kwargs=grid_fit_kw,
            is_opf=cfg['is_opf'],
        )

        # Record grid search best scores
        gridsearch_results_f1[model_name].append(best_f1)
        gridsearch_results_acc[model_name].append(best_acc)

        # --- Final training on full training set ---
        final_kw = cfg['final_fit_kwargs'].copy()

        if model_name == 'MLP':
            final_kw['sample_weight'] = sw_train_full
        if model_name == 'XGB':
            final_kw['scale_pos_weight'] = counts_full[0] / counts_full[1]

        # Separate sample_weight from constructor args for MLP
        final_sample_weight = final_kw.pop('sample_weight', None)
        final_model = cfg['class'](**{**best_params, **final_kw})

        if cfg['is_opf']:
            final_model.fit(np.array(X_train_scaled), np.array(y_train, dtype=np.int32))
        else:
            if final_sample_weight is not None:
                final_model.fit(X_train_scaled, y_train, sample_weight=final_sample_weight)
            else:
                final_model.fit(X_train_scaled, y_train)

        # --- Evaluation on test set ---
        y_pred = final_model.predict(np.array(X_test_scaled) if cfg['is_opf'] else X_test_scaled)

        results[model_name]['accuracy'].append(accuracy_score(y_test, y_pred))
        results[model_name]['precision'].append(precision_score(y_test, y_pred, average='weighted'))
        results[model_name]['recall'].append(recall_score(y_test, y_pred, average='weighted'))
        results[model_name]['f1'].append(f1_score(y_test, y_pred, average='weighted'))

        if cfg['supports_proba']:
            y_proba = final_model.predict_proba(np.array(X_test_scaled) if cfg['is_opf'] else X_test_scaled)[:, 1]
            results[model_name]['auroc'].append(roc_auc_score(y_test, y_proba))
        else:
            results[model_name]['auroc'].append(roc_auc_score(y_test, y_pred))

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        confusion_matrices[model_name].append(cm)

        # MDI for tree-based models
        if cfg['tracks_mdi'] and not cfg['is_opf']:
            mdi_results[model_name].append(final_model.feature_importances_)

        # Print per-fold results
        param_display = ', '.join(f'{k}={v}' for k, v in list(best_params.items())[:2])
        print(f'  {model_name:>4} | Best params: {param_display} | '
              f'Acc: {results[model_name]["accuracy"][-1]:.4f} | '
              f'F1: {results[model_name]["f1"][-1]:.4f} | '
              f'AUROC: {results[model_name]["auroc"][-1]:.4f}')

    print("-" * 50)


## 9. GridSearch Results


In [ ]:
# GridSearch — F1 score summary across folds
df_gs = pd.DataFrame(gridsearch_results_f1,
                      index=[f'Fold {i+1}' for i in range(len(gridsearch_results_f1['KNN']))])

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

for model in df_gs.columns:
    axes[0].plot(df_gs.index, df_gs[model], marker='o', label=model, linewidth=2)
axes[0].set_title('Best Validation F1 per Fold (Grid Search)')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('F1 Score')
axes[0].legend()
axes[0].grid(True, alpha=0.4)
axes[0].set_ylim(0, 1)

means = df_gs.mean()
stds = df_gs.std()
bars = axes[1].bar(means.index, means.values, yerr=stds.values,
                   capsize=5, color=sns.color_palette('viridis', len(means)), alpha=0.85)
for bar, mean, std in zip(bars, means.values, stds.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01,
                 f'{mean:.3f}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Mean ± Std F1 (Grid Search)')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Mean F1 Score')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.4, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# GridSearch — Accuracy summary across folds
df_gs_acc = pd.DataFrame(gridsearch_results_acc,
                          index=[f'Fold {i+1}' for i in range(len(gridsearch_results_acc['KNN']))])

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

for model in df_gs_acc.columns:
    axes[0].plot(df_gs_acc.index, df_gs_acc[model], marker='o', label=model, linewidth=2)
axes[0].set_title('Best Validation Accuracy per Fold (Grid Search)')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.4)
axes[0].set_ylim(0, 1)

means = df_gs_acc.mean()
stds = df_gs_acc.std()
bars = axes[1].bar(means.index, means.values, yerr=stds.values,
                   capsize=5, color=sns.color_palette('viridis', len(means)), alpha=0.85)
for bar, mean, std in zip(bars, means.values, stds.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01,
                 f'{mean:.3f}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Mean ± Std Accuracy (Grid Search)')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Mean Accuracy')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.4, axis='y')

plt.tight_layout()
plt.show()


## 10. Final Results and Metrics


In [ ]:
# Mapping from short model keys to display names
MODEL_DISPLAY_NAMES = {
    'KNN': 'KNeighborsClassifier',
    'DT': 'DecisionTreeClassifier',
    'RF': 'RandomForestClassifier',
    'SVM': 'SVM',
    'MLP': 'MLPClassifier',
    'LR': 'LogisticRegression',
    'XGB': 'XGBClassifier',
    'NB': 'GaussianNB',
    'NC': 'NearestCentroid',
    'OPF': 'SupervisedOPF',
}

METRIC_DISPLAY = {
    'accuracy': 'Accuracy',
    'precision': 'Precision',
    'recall': 'Recall',
    'f1': 'F1-Score',
    'auroc': 'AUROC',
}

print("\nFINAL RESULTS — MEAN ± STANDARD DEVIATION")

for model in MODEL_NAMES:
    name = MODEL_DISPLAY_NAMES[model]
    print(f'\n{name}:')
    for metric in METRICS:
        vals = results[model][metric]
        print(f'  {METRIC_DISPLAY[metric]:>10}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')


## 11. Confidence Intervals


In [ ]:
def get_confidence_interval(data, confidence=0.95):
    """Compute 95% confidence interval half-width using t-distribution."""
    data = np.array(data)
    n = len(data)
    if n <= 1:
        return 0.0
    se = st.sem(data)
    h = se * st.t.ppf((1 + confidence) / 2., n - 1)
    return h if not np.isnan(h) else 0.0

# Build algorithms dict for CI plotting (compatible with original code)
algorithms_ci = {}
for model in MODEL_NAMES:
    name = MODEL_DISPLAY_NAMES[model]
    algorithms_ci[name] = (
        results[model]['accuracy'],
        results[model]['f1'],
        results[model]['precision'],
        results[model]['recall'],
        results[model]['auroc'],
    )

metric_names = ['Accuracy', 'F1-Score', 'Precision', 'Recall', 'AUROC']
model_names = list(algorithms_ci.keys())

for i, metric in enumerate(metric_names):
    data_for_metric = {model: algorithms_ci[model][i] for model in model_names}
    num_folds = len(list(data_for_metric.values())[0])
    df_metric = pd.DataFrame(data_for_metric, index=[f'Fold {j+1}' for j in range(num_folds)])

    lower_ci = df_metric.mean() - df_metric.apply(lambda x: get_confidence_interval(x.dropna()))
    min_y_limit = lower_ci.min() - 0.05
    min_y_limit = max(0, min_y_limit)

    fig, axes = plt.subplots(1, 2, figsize=(20, 6))

    # Per-fold line plot
    for model in df_metric.columns:
        axes[0].plot(df_metric.index, df_metric[model], marker='o', label=model, linewidth=2)
    axes[0].set_title(f'Performance by Fold — {metric}', fontsize=14)
    axes[0].set_xlabel('Fold')
    axes[0].set_ylabel(metric)
    axes[0].legend()
    axes[0].grid(True, alpha=0.4)
    axes[0].set_ylim(min_y_limit, 1.0)
    axes[0].set_xticks(range(len(df_metric.index)))
    axes[0].set_xticklabels(df_metric.index, rotation=45, ha='right')

    # Mean ± CI bar plot
    means = df_metric.mean()
    cis = df_metric.apply(lambda x: get_confidence_interval(x.dropna()))
    bars = axes[1].bar(means.index, means.values, yerr=cis.values,
                       capsize=5, color=sns.color_palette('viridis', len(means)), alpha=0.85)
    for bar, mean, ci in zip(bars, means.values, cis.values):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + ci + 0.01,
                     f'{mean:.3f}', ha='center', fontsize=9, fontweight='bold')
    axes[1].set_title(f'Mean with 95% CI — {metric}', fontsize=14)
    axes[1].set_xlabel('Model')
    axes[1].set_ylabel(f'Mean {metric}')
    axes[1].grid(True, alpha=0.4, axis='y')
    axes[1].set_ylim(min_y_limit, 1.0)
    axes[1].set_xticks(range(len(means.index)))
    axes[1].set_xticklabels(means.index, rotation=45, ha='right', fontsize=11)

    plt.tight_layout()
    plt.show()


## 12. Confusion Matrices


In [ ]:
# Accumulated confusion matrices (sum across folds for each model)
plt.figure(figsize=(20, 16))
names_display = ['KNN', 'Decision Tree', 'Random Forest', 'SVM', 'MLP',
                 'Logistic Regression', 'XGBoost', 'GaussianNB', 'NearestCentroid', 'SupervisedOPF']
cmap_colors = ['Blues', 'Greens', 'Oranges', 'Reds', 'Purples', 'Greys', 'YlOrBr', 'YlGnBu', 'BuPu', 'YlOrRd']

for j, (model, disp_name, cmap) in enumerate(zip(MODEL_NAMES, names_display, cmap_colors)):
    # Use last fold's confusion matrix for display
    cm = confusion_matrices[model][-1] if confusion_matrices[model] else np.zeros((2, 2))
    plt.subplot(3, 4, j + 1)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['0-1 Visit', '2+ Visits'])
    disp.plot(cmap=cmap, ax=plt.gca(), values_format='d')
    plt.title(f'{disp_name}', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted', fontsize=9)
    plt.ylabel('Actual', fontsize=9)

plt.suptitle('Confusion Matrices — Medical Visits (Last Fold)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()


## 13. Wilcoxon Statistical Test


In [ ]:
# Build all_metrics_results for Wilcoxon pairwise comparisons
all_metrics_results = {
    'Accuracy': [],
    'Precision': [],
    'Recall': [],
    'F1-Score': [],
    'AUROC': [],
}

metric_keys = ['accuracy', 'precision', 'recall', 'f1', 'auroc']
display_metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUROC']

for model in MODEL_NAMES:
    name = MODEL_DISPLAY_NAMES[model]
    for mkey, dkey in zip(metric_keys, display_metrics):
        all_metrics_results[dkey].append((name, results[model][mkey]))

alpha = 0.05

for metric_name, model_scores_list in all_metrics_results.items():
    for i in range(len(model_scores_list)):
        for j in range(i + 1, len(model_scores_list)):
            model1_name, scores1 = model_scores_list[i]
            model2_name, scores2 = model_scores_list[j]

            print(f'\nComparing {model1_name} vs {model2_name}:')

            if len(scores1) != len(scores2):
                print('\t- ERROR: Score list lengths differ.')
                continue
            if len(scores1) < 2:
                print('\t- ERROR: Not enough scores for the test.')
                continue

            try:
                stat, p_val = wilcoxon(scores1, scores2, zero_method='zsplit', correction=False)
            except ValueError as e:
                print(f'\t- WARNING: Wilcoxon test could not be computed. ({e})')
                mean1 = np.mean(scores1)
                mean2 = np.mean(scores2)
                print(f'\t- Mean {model1_name}: {mean1:.4f}')
                print(f'\t- Mean {model2_name}: {mean2:.4f}')
                print('\t-> No statistical conclusion.')
                print("-" * 50)
                continue

            mean1 = np.mean(scores1)
            mean2 = np.mean(scores2)

            print(f'\t- n: {len(scores1)}')
            print(f'\t- Mean {model1_name}: {mean1:.4f}')
            print(f'\t- Mean {model2_name}: {mean2:.4f}')
            print(f'\t- Z statistic: {stat:.4f}')
            print(f'\t- p-value: {p_val:.4f}')

            if p_val < alpha:
                print(f'\t-> STATISTICALLY SIGNIFICANT (p < {alpha}).')
                best_model = model1_name if mean1 > mean2 else model2_name
                print(f'\t-> Best model (by mean): {best_model}')
            else:
                print(f'\t-> No statistically significant difference (p >= {alpha}).')
            print("-" * 50)
